# Custom CNN — LIDC nodule malignancy + Grad-CAM

Slot: `custom_cnn` (lightweight `ImageCNN`).

Pipeline: load shared crops/splits → train image-only CNN → evaluate → Grad-CAM overlays.

Shared data lives under `outputs/`; artifacts go to `models/custom_cnn/` and `gradcam/custom_cnn/`.


In [ ]:
from pathlib import Path
import os
import random
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)
from sklearn.model_selection import GroupShuffleSplit

_PROJECT_ROOT = Path("..").resolve()
if not (_PROJECT_ROOT / "outputs").exists():
    _PROJECT_ROOT = Path(".").resolve()

_SRC = _PROJECT_ROOT / "src"
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

SLOT_ID = "custom_cnn"

OUTPUT_ROOT = _PROJECT_ROOT / "outputs"
MODELS_ROOT = _PROJECT_ROOT / "models"
GRADCAM_ROOT = _PROJECT_ROOT / "gradcam"

PREPROCESS_DIR = OUTPUT_ROOT / "preprocessing"
CROP_DIR = OUTPUT_ROOT / "crops"
SPLITS_DIR = OUTPUT_ROOT / "splits"

MODEL_DIR = MODELS_ROOT / SLOT_ID
GRADCAM_DIR = GRADCAM_ROOT / SLOT_ID

DATA_CSV = PREPROCESS_DIR / "lidc_model_table_fixed_crops.csv"
TRAIN_DF_CSV = SPLITS_DIR / "train_df.csv"
VAL_DF_CSV = SPLITS_DIR / "val_df.csv"
TEST_DF_CSV = SPLITS_DIR / "test_df.csv"

BEST_MODEL_PATH = MODEL_DIR / "best_model.pth"
HISTORY_PATH = MODEL_DIR / "training_history.csv"
TEST_PRED_PATH = MODEL_DIR / "test_predictions.csv"
TEST_RESULTS_PATH = MODEL_DIR / "test_results.csv"
MEAN_PATH = MODEL_DIR / "image_mean.npy"
STD_PATH = MODEL_DIR / "image_std.npy"

for d in [MODEL_DIR, GRADCAM_DIR, PREPROCESS_DIR, CROP_DIR, SPLITS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PROJECT_ROOT:", _PROJECT_ROOT)
print("SLOT_ID:", SLOT_ID)
print("MODEL_DIR:", MODEL_DIR)
print("GRADCAM_DIR:", GRADCAM_DIR)
print("Device:", DEVICE)


## 1. Load table and patient-level splits


In [ ]:
image_col = "crop_path"
target_col = "label"


def fix_crop_path(path: str) -> str:
    raw = str(path).replace("\\", "/")
    raw = raw.replace("/outputs/crops_fixed_v2/", "/outputs/crops/")
    p = Path(raw)
    if p.exists():
        return str(p)
    hits = list(CROP_DIR.rglob(Path(raw).name))
    return str(hits[0]) if hits else str(CROP_DIR / Path(raw).name)


def load_or_make_splits():
    if TRAIN_DF_CSV.exists() and VAL_DF_CSV.exists() and TEST_DF_CSV.exists():
        train_df = pd.read_csv(TRAIN_DF_CSV)
        val_df = pd.read_csv(VAL_DF_CSV)
        test_df = pd.read_csv(TEST_DF_CSV)
        print("Loaded existing splits from", SPLITS_DIR)
    else:
        if not DATA_CSV.exists():
            raise FileNotFoundError(
                f"Missing {DATA_CSV}. Run preprocessing / expand_lidc first."
            )
        df = pd.read_csv(DATA_CSV)
        gss1 = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=SEED)
        train_val_idx, test_idx = next(
            gss1.split(df, df[target_col], groups=df["patient_id"])
        )
        train_val_df = df.iloc[train_val_idx].reset_index(drop=True)
        test_df = df.iloc[test_idx].reset_index(drop=True)
        gss2 = GroupShuffleSplit(n_splits=1, test_size=0.1765, random_state=SEED)
        train_idx, val_idx = next(
            gss2.split(
                train_val_df,
                train_val_df[target_col],
                groups=train_val_df["patient_id"],
            )
        )
        train_df = train_val_df.iloc[train_idx].reset_index(drop=True)
        val_df = train_val_df.iloc[val_idx].reset_index(drop=True)
        train_df.to_csv(TRAIN_DF_CSV, index=False)
        val_df.to_csv(VAL_DF_CSV, index=False)
        test_df.to_csv(TEST_DF_CSV, index=False)
        print("Created splits and saved to", SPLITS_DIR)

    for frame in (train_df, val_df, test_df):
        frame[image_col] = frame[image_col].map(fix_crop_path)

    cleaned = []
    for name, frame in [("train", train_df), ("val", val_df), ("test", test_df)]:
        ok = frame[image_col].map(lambda p: Path(p).exists())
        if (~ok).any():
            print(f"[warn] dropping {(~ok).sum()} {name} rows with missing crops")
            frame = frame.loc[ok].reset_index(drop=True)
        cleaned.append(frame)
    return cleaned


train_df, val_df, test_df = load_or_make_splits()
print("Train/Val/Test:", train_df.shape, val_df.shape, test_df.shape)
print(train_df[target_col].value_counts())


## 2. Image stats, dataset, model


In [ ]:
def load_crop(path):
    crop = np.load(path).astype(np.float32)
    if crop.shape != (64, 64):
        raise ValueError(f"Unexpected crop shape {crop.shape} for {path}")
    return crop


if MEAN_PATH.exists() and STD_PATH.exists():
    img_mean = float(np.load(MEAN_PATH))
    img_std = float(np.load(STD_PATH))
    print("Loaded mean/std from", MODEL_DIR)
else:
    train_pixels = np.stack([load_crop(p) for p in train_df[image_col].values])
    img_mean = float(train_pixels.mean())
    img_std = float(train_pixels.std())
    if img_std < 1e-6:
        img_std = 1.0
    np.save(MEAN_PATH, img_mean)
    np.save(STD_PATH, img_std)
    print("Computed and saved mean/std")

print("mean:", img_mean, "std:", img_std)


class ImageOnlyDataset(Dataset):
    def __init__(self, df, image_col, target_col, mean, std, augment=False):
        self.df = df.reset_index(drop=True)
        self.image_col = image_col
        self.target_col = target_col
        self.mean = mean
        self.std = std
        self.augment = augment

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = np.load(row[self.image_col]).astype(np.float32)
        if self.augment:
            if np.random.rand() < 0.5:
                img = np.fliplr(img).copy()
            if np.random.rand() < 0.5:
                img = np.flipud(img).copy()
        img = (img - self.mean) / self.std
        img = torch.tensor(img, dtype=torch.float32).unsqueeze(0)
        label = torch.tensor(row[self.target_col], dtype=torch.float32)
        return img, label


BATCH_SIZE = 32
train_loader = DataLoader(
    ImageOnlyDataset(train_df, image_col, target_col, img_mean, img_std, augment=True),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
)
val_loader = DataLoader(
    ImageOnlyDataset(val_df, image_col, target_col, img_mean, img_std, augment=False),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
)
test_loader = DataLoader(
    ImageOnlyDataset(test_df, image_col, target_col, img_mean, img_std, augment=False),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
)
print("batches train/val/test:", len(train_loader), len(val_loader), len(test_loader))


class ImageCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
        )
        self.classifier = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.50),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        return self.classifier(self.encoder(x)).squeeze(1)


model = ImageCNN().to(DEVICE)
print("params:", sum(p.numel() for p in model.parameters() if p.requires_grad))


## 3. Train (skipped automatically if `best_model.pth` exists)


In [ ]:
SKIP_TRAIN = BEST_MODEL_PATH.exists()  # set False to force retrain

y_train = train_df[target_col].values.astype(np.float32)
num_neg = int((y_train == 0).sum())
num_pos = max(int((y_train == 1).sum()), 1)
pos_weight = torch.tensor([num_neg / num_pos], dtype=torch.float32).to(DEVICE)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=10
)


def compute_metrics(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    acc = accuracy_score(y_true, y_pred)
    auc = roc_auc_score(y_true, y_prob) if len(np.unique(y_true)) > 1 else float("nan")
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    return {
        "accuracy": acc,
        "auc": auc,
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "sensitivity": recall_score(y_true, y_pred, zero_division=0),
        "specificity": tn / (tn + fp) if (tn + fp) > 0 else 0.0,
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }


def run_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    total_loss, all_probs, all_labels = 0.0, [], []
    with torch.set_grad_enabled(is_train):
        for img_batch, y_batch in loader:
            img_batch = img_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE)
            logits = model(img_batch)
            loss = criterion(logits, y_batch)
            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            probs = torch.sigmoid(logits)
            total_loss += loss.item() * img_batch.size(0)
            all_probs.extend(probs.detach().cpu().numpy())
            all_labels.extend(y_batch.detach().cpu().numpy())
    return total_loss / len(loader.dataset), np.array(all_probs), np.array(all_labels)


if SKIP_TRAIN:
    print("SKIP_TRAIN=True — using existing", BEST_MODEL_PATH)
else:
    EPOCHS, PATIENCE = 100, 20
    best_val_auc = -np.inf
    wait = 0
    history = []
    for epoch in range(1, EPOCHS + 1):
        train_loss, train_probs, train_labels = run_epoch(
            model, train_loader, criterion, optimizer
        )
        val_loss, val_probs, val_labels = run_epoch(model, val_loader, criterion, None)
        train_m = compute_metrics(train_labels, train_probs)
        val_m = compute_metrics(val_labels, val_probs)
        scheduler.step(val_m["auc"])
        history.append(
            {
                "epoch": epoch,
                "train_loss": train_loss,
                "val_loss": val_loss,
                "train_auc": train_m["auc"],
                "val_auc": val_m["auc"],
                "train_accuracy": train_m["accuracy"],
                "val_accuracy": val_m["accuracy"],
                "train_f1": train_m["f1"],
                "val_f1": val_m["f1"],
            }
        )
        if val_m["auc"] > best_val_auc:
            best_val_auc = val_m["auc"]
            wait = 0
            torch.save(model.state_dict(), BEST_MODEL_PATH)
            tag = " [SAVED]"
        else:
            wait += 1
            tag = ""
        if epoch == 1 or epoch % 10 == 0 or tag:
            print(
                f"Epoch {epoch:03d} | train_loss={train_loss:.4f} "
                f"val_loss={val_loss:.4f} val_auc={val_m['auc']:.4f}{tag}"
            )
        if wait >= PATIENCE:
            print(f"Early stopping at epoch {epoch}")
            break
    pd.DataFrame(history).to_csv(HISTORY_PATH, index=False)
    print("Best val AUC:", best_val_auc)
    print("Saved history:", HISTORY_PATH)


## 4. Test evaluation


In [ ]:
best_model = ImageCNN().to(DEVICE)
best_model.load_state_dict(
    torch.load(BEST_MODEL_PATH, map_location=DEVICE, weights_only=True)
)
best_model.eval()

test_loss, test_probs, test_labels = run_epoch(best_model, test_loader, criterion, None)
test_metrics = compute_metrics(test_labels, test_probs, threshold=0.5)
print("Test metrics:", test_metrics)
print(
    classification_report(
        test_labels,
        (test_probs >= 0.5).astype(int),
        target_names=["Benign", "Malignant"],
    )
)

keep_cols = [c for c in ["patient_id", "malignancy_mean", image_col, target_col] if c in test_df.columns]
pred_df = test_df[keep_cols].copy().reset_index(drop=True)
pred_df["prob"] = test_probs
pred_df["pred"] = (test_probs >= 0.5).astype(int)
pred_df["correct"] = pred_df["pred"] == pred_df[target_col]
pred_df.to_csv(TEST_PRED_PATH, index=False)

results = pd.DataFrame([{"model": SLOT_ID, "threshold": 0.5, **test_metrics}])
results.to_csv(TEST_RESULTS_PATH, index=False)
print("Saved:", TEST_PRED_PATH)
print("Saved:", TEST_RESULTS_PATH)
display(results)


## 5. Grad-CAM


In [ ]:
from xai import GradCAM, plot_gradcam_grid

# Last Conv2d before pooling in encoder: index 8
target_layer = best_model.encoder[8]
print("target_layer:", target_layer)

grad_cam = GradCAM(best_model, target_layer)


def prepare_gradcam_input(row):
    raw_img = np.load(row[image_col]).astype(np.float32)
    img = (raw_img - img_mean) / img_std
    img_tensor = (
        torch.tensor(img, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(DEVICE)
    )
    return img_tensor, raw_img


threshold = 0.5
gdf = pred_df.copy()
gdf["label"] = gdf[target_col].astype(int)
gdf["pred"] = (gdf["prob"] >= threshold).astype(int)


def case_type(row):
    return {(0, 0): "TN", (1, 1): "TP", (0, 1): "FP", (1, 0): "FN"}[
        (int(row.label), int(row.pred))
    ]


gdf["case_type"] = gdf.apply(case_type, axis=1)

picked = []
for t in ["TN", "TP", "FP", "FN"]:
    sub = gdf[gdf["case_type"] == t]
    if t in {"TN", "FN"}:
        sub = sub.sort_values("prob", ascending=True)
    else:
        sub = sub.sort_values("prob", ascending=False)
    n = 1 if t in {"TN", "TP"} else 2
    if len(sub):
        picked.append(sub.head(n))

cases = pd.concat(picked).reset_index(drop=True) if picked else gdf.head(4).copy()
cases_path = GRADCAM_DIR / "gradcam_selected_cases.csv"
cases.to_csv(cases_path, index=False)
print("Selected cases:")
display(
    cases[
        [
            c
            for c in ["case_type", "patient_id", "label", "prob", "pred", "malignancy_mean"]
            if c in cases.columns
        ]
    ]
)

plot_gradcam_grid(
    cases,
    prepare_fn=prepare_gradcam_input,
    generate_fn=grad_cam.generate,
    title=f"Grad-CAM | {SLOT_ID}",
    save_path=str(GRADCAM_DIR / "gradcam_selected_cases.png"),
    show=True,
)

main_cases = []
for t in ["TN", "TP", "FP", "FN"]:
    sub = cases[cases["case_type"] == t]
    if len(sub):
        main_cases.append(sub.iloc[0])
if main_cases:
    main_df = pd.DataFrame(main_cases).reset_index(drop=True)
    plot_gradcam_grid(
        main_df,
        prepare_fn=prepare_gradcam_input,
        generate_fn=grad_cam.generate,
        title=f"Grad-CAM (4 cases) | {SLOT_ID}",
        save_path=str(GRADCAM_DIR / "gradcam_4_cases.png"),
        show=True,
    )

grad_cam.close()
print("Grad-CAM saved under", GRADCAM_DIR)
